
# Stability Screening and Localization Sensitivity Analysis

## 1. Research Objectives

This notebook extends the stability–localization disagreement audit by evaluating whether explanation similarity can serve as a reliable screening signal for substantial localization deterioration.

Two research questions guide the analysis:

**RQ1: Screening Reliability**

How effectively does explanation similarity distinguish evaluations with substantial localization alignment loss from those without it? How many localization-loss cases would be missed when high explanation similarity is treated as a reassuring signal?

**RQ2: Localization Sensitivity**

How sensitive are the findings to the definition of substantial localization loss, and how does annotated opacity-region size influence the interpretation of localization deterioration?

The analysis uses precomputed evaluation metrics from 150 annotated chest X-ray images, three XAI methods (Integrated Gradients, Occlusion Sensitivity, and Score-CAM), and six image-corruption conditions.

All screening results are retrospective and exploratory. Localization alignment is an attribution-based proxy and does not establish clinical explanation correctness or faithfulness.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

data_path = Path(
    "/content/drive/MyDrive/XAI_project_Expo_1_folder/"
    "Results/ExpandedEvaluation/"
    "stability_localization_area_enriched.csv"
)

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst three rows:")
display(df.head(3))

print("\nMissing values:")
print(df.isna().sum()[df.isna().sum() > 0])

Dataset shape: (2700, 24)

Column names:
['sop_uid', 'true_class', 'original_predicted_class', 'original_correct', 'corrupted_predicted_class', 'prediction_changed', 'corruption', 'severity', 'sigma', 'opacity_probability', 'explanation_similarity', 'localization_alignment', 'n_boxes', 'convergence_delta', 'method', 'original_alignment', 'alignment_change', 'high_stability', 'localization_loss', 'disagreement', 'box_area_fraction', 'attribution_enrichment', 'original_enrichment', 'enrichment_change']

First three rows:


,sop_uid,true_class,original_predicted_class,original_correct,corrupted_predicted_class,prediction_changed,corruption,severity,sigma,opacity_probability,...,method,original_alignment,alignment_change,high_stability,localization_loss,disagreement,box_area_fraction,attribution_enrichment,original_enrichment,enrichment_change
0,1.2.276.0.7230010.3.1.4.8323329.10261.15178743...,1,1,1,1,0,Blur,mild,0.5,0.545375,...,Integrated Gradients,0.124,-0.009625,True,False,False,0.041162,2.778629,3.012471,-0.233843
1,1.2.276.0.7230010.3.1.4.8323329.10261.15178743...,1,1,1,1,0,Blur,moderate,1.5,0.535043,...,Integrated Gradients,0.124,-0.020098,False,False,False,0.041162,2.524204,3.012471,-0.488267
2,1.2.276.0.7230010.3.1.4.8323329.10261.15178743...,1,1,1,1,0,Blur,severe,3.0,0.569804,...,Integrated Gradients,0.124,0.013435,False,False,False,0.041162,3.338858,3.012471,0.326387



Missing values:
convergence_delta    1800
dtype: int64


In [ ]:
# Validate the evaluation dataset before analysis

required_columns = [
    "sop_uid",
    "method",
    "corruption",
    "severity",
    "explanation_similarity",
    "original_alignment",
    "localization_alignment",
    "alignment_change",
    "box_area_fraction",
]

assert (
    not df[required_columns].isna().any().any()
), "Missing values detected in required columns."

assert len(df) == 2700, "Unexpected evaluation count."
assert df["sop_uid"].nunique() == 150
assert df["method"].nunique() == 3

keys = ["sop_uid", "method", "corruption", "severity"]

assert not df.duplicated(keys).any(), "Duplicate evaluation combinations detected."

assert (
    df.groupby("sop_uid").size() == 18
).all(), "Some images do not have 18 evaluations."

assert np.allclose(
    df["alignment_change"],
    df["localization_alignment"] - df["original_alignment"],
    atol=1e-8,
), "Alignment change is inconsistent."

print("Dataset validation passed.")
print("Images:", df["sop_uid"].nunique())
print("Evaluations:", len(df))

Dataset validation passed.
Images: 150
Evaluations: 2700


## 2. Screening Reliability Analysis

In [ ]:
# Experiment 1: Define the screening outcome

# A substantial localization loss means alignment
# decreased by at least 5 percentage points.
df["substantial_loss"] = df["alignment_change"] <= -0.05

# Lower explanation similarity indicates greater
# explanation instability.
df["instability_score"] = -df["explanation_similarity"]

print("Total evaluations:", len(df))
print("Unique images:", df["sop_uid"].nunique())

print("\nSubstantial localization loss:")
print(df["substantial_loss"].value_counts())

print("\nLocalization-loss prevalence:")
print(f"{df['substantial_loss'].mean() * 100:.2f}%")

print("\nLoss prevalence by XAI method:")
display(
    df.groupby("method")["substantial_loss"]
    .agg(["sum", "count", "mean"])
    .assign(loss_percentage=lambda x: x["mean"] * 100)
)

Total evaluations: 2700
Unique images: 150

Substantial localization loss:
substantial_loss
False    1555
True     1145
Name: count, dtype: int64

Localization-loss prevalence:
42.41%

Loss prevalence by XAI method:


,sum,count,mean,loss_percentage
method,,,,
Integrated Gradients,425,900,0.472222,47.222222
Occlusion Sensitivity,397,900,0.441111,44.111111
Score-CAM,323,900,0.358889,35.888889


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

y_true = df["substantial_loss"].astype(int)
y_score = df["instability_score"]

roc_auc = roc_auc_score(y_true, y_score)
avg_precision = average_precision_score(y_true, y_score)

print(f"ROC-AUC: {roc_auc:.4f}")
print(f"Average precision: {avg_precision:.4f}")
print(f"Event prevalence: {y_true.mean():.4f}")

ROC-AUC: 0.7911
Average precision: 0.7231
Event prevalence: 0.4241


In [ ]:
from sklearn.metrics import confusion_matrix

results = []

for threshold in [0.60, 0.70, 0.80, 0.90]:
    flagged = df["explanation_similarity"] < threshold

    tn, fp, fn, tp = confusion_matrix(
        df["substantial_loss"], flagged, labels=[False, True]
    ).ravel()

    results.append(
        {
            "similarity_threshold": threshold,
            "sensitivity": tp / (tp + fn),
            "specificity": tn / (tn + fp),
            "precision": tp / (tp + fp) if tp + fp else np.nan,
            "missed_loss_cases": fn,
            "missed_loss_pct": 100 * fn / (tp + fn),
            "flagged_evaluations": tp + fp,
        }
    )

screening_results = pd.DataFrame(results)

display(screening_results.round(3))

,similarity_threshold,sensitivity,specificity,precision,missed_loss_cases,missed_loss_pct,flagged_evaluations
0,0.6,0.567,0.806,0.683,496,43.319,950
1,0.7,0.719,0.687,0.629,322,28.122,1309
2,0.8,0.851,0.582,0.600,171,14.934,1624
3,0.9,0.966,0.320,0.511,39,3.406,2164


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np
import pandas as pd

# Reproducible image-cluster bootstrap
rng = np.random.default_rng(42)

n_bootstrap = 2000
image_ids = df["sop_uid"].unique()

# Pre-group rows by image to preserve within-image dependence
image_groups = {uid: group for uid, group in df.groupby("sop_uid")}

bootstrap_results = []

for b in range(n_bootstrap):

    # Resample images with replacement
    sampled_ids = rng.choice(image_ids, size=len(image_ids), replace=True)

    # Include all evaluations for each sampled image
    boot_df = pd.concat([image_groups[uid] for uid in sampled_ids], ignore_index=True)

    y_true = boot_df["substantial_loss"].astype(int)
    y_score = -boot_df["explanation_similarity"]

    # ROC-AUC requires both outcome classes
    if y_true.nunique() < 2:
        continue

    auc = roc_auc_score(y_true, y_score)
    ap = average_precision_score(y_true, y_score)

    # Missed localization losses at similarity >= 0.80
    loss_cases = boot_df["substantial_loss"]

    missed = loss_cases & (boot_df["explanation_similarity"] >= 0.80)

    missed_pct = 100 * missed.sum() / loss_cases.sum()

    bootstrap_results.append(
        {"roc_auc": auc, "average_precision": ap, "missed_loss_pct_080": missed_pct}
    )

boot_results = pd.DataFrame(bootstrap_results)

summary = pd.DataFrame(
    {
        "Metric": ["ROC-AUC", "Average precision", "Missed loss (%) at 0.80"],
        "Bootstrap mean": [
            boot_results["roc_auc"].mean(),
            boot_results["average_precision"].mean(),
            boot_results["missed_loss_pct_080"].mean(),
        ],
        "CI lower (2.5%)": [
            boot_results["roc_auc"].quantile(0.025),
            boot_results["average_precision"].quantile(0.025),
            boot_results["missed_loss_pct_080"].quantile(0.025),
        ],
        "CI upper (97.5%)": [
            boot_results["roc_auc"].quantile(0.975),
            boot_results["average_precision"].quantile(0.975),
            boot_results["missed_loss_pct_080"].quantile(0.975),
        ],
    }
)

print("Valid bootstrap samples:", len(boot_results))
display(summary.round(4))

Valid bootstrap samples: 2000


,Metric,Bootstrap mean,CI lower (2.5%),CI upper (97.5%)
0,ROC-AUC,0.7914,0.7671,0.8145
1,Average precision,0.7234,0.6705,0.7726
2,Missed loss (%) at 0.80,14.9317,12.1185,17.8147


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

method_results = []

for method, group in df.groupby("method"):

    y_true = group["substantial_loss"].astype(int)
    y_score = -group["explanation_similarity"]

    # Similarity below 0.80 triggers review
    flagged = group["explanation_similarity"] < 0.80

    tp = (flagged & group["substantial_loss"]).sum()
    fn = (~flagged & group["substantial_loss"]).sum()
    fp = (flagged & ~group["substantial_loss"]).sum()
    tn = (~flagged & ~group["substantial_loss"]).sum()

    method_results.append(
        {
            "method": method,
            "n_evaluations": len(group),
            "loss_prevalence_pct": 100 * y_true.mean(),
            "roc_auc": roc_auc_score(y_true, y_score),
            "average_precision": average_precision_score(y_true, y_score),
            "sensitivity_080": tp / (tp + fn),
            "specificity_080": tn / (tn + fp),
            "missed_loss_cases_080": fn,
            "missed_loss_pct_080": 100 * fn / (tp + fn),
            "flagged_pct_080": 100 * (tp + fp) / len(group),
        }
    )

method_screening = pd.DataFrame(method_results)

display(method_screening.round(4))

,method,n_evaluations,loss_prevalence_pct,roc_auc,average_precision,sensitivity_080,specificity_080,missed_loss_cases_080,missed_loss_pct_080,flagged_pct_080
0,Integrated Gradients,900,47.2222,0.7850,0.7525,0.9718,0.3368,12,2.8235,80.8889
1,Occlusion Sensitivity,900,44.1111,0.7150,0.5943,0.7431,0.5924,102,25.6927,55.5556
2,Score-CAM,900,35.8889,0.8821,0.8123,0.8235,0.7747,57,17.6471,44.0000


In [ ]:
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

methods = ["Integrated Gradients", "Occlusion Sensitivity", "Score-CAM"]

image_ids = df["sop_uid"].unique()

# Reuse the image groups from Step 4
bootstrap_auc = []

for b in range(2000):
    sampled_ids = rng.choice(image_ids, size=len(image_ids), replace=True)

    boot_df = pd.concat([image_groups[uid] for uid in sampled_ids], ignore_index=True)

    result = {}

    for method in methods:
        subset = boot_df[boot_df["method"] == method]

        y_true = subset["substantial_loss"].astype(int)
        y_score = -subset["explanation_similarity"]

        if y_true.nunique() < 2:
            result[method] = np.nan
        else:
            result[method] = roc_auc_score(y_true, y_score)

    bootstrap_auc.append(result)

auc_boot_df = pd.DataFrame(bootstrap_auc)

auc_summary = pd.DataFrame(
    {
        "Method": methods,
        "ROC-AUC": [
            method_screening.loc[method_screening["method"] == m, "roc_auc"].iloc[0]
            for m in methods
        ],
        "CI lower": [auc_boot_df[m].quantile(0.025) for m in methods],
        "CI upper": [auc_boot_df[m].quantile(0.975) for m in methods],
    }
)

print("Method-specific ROC-AUC confidence intervals:")
display(auc_summary.round(4))

print("\nPaired bootstrap differences:")

comparisons = [
    ("Score-CAM", "Integrated Gradients"),
    ("Score-CAM", "Occlusion Sensitivity"),
    ("Integrated Gradients", "Occlusion Sensitivity"),
]

for a, b in comparisons:
    difference = auc_boot_df[a] - auc_boot_df[b]

    print(
        f"{a} minus {b}: "
        f"mean={difference.mean():.4f}, "
        f"95% CI=[{difference.quantile(0.025):.4f}, "
        f"{difference.quantile(0.975):.4f}]"
    )

Method-specific ROC-AUC confidence intervals:


,Method,ROC-AUC,CI lower,CI upper
0,Integrated Gradients,0.7850,0.7456,0.8240
1,Occlusion Sensitivity,0.7150,0.6702,0.7586
2,Score-CAM,0.8821,0.8594,0.9037



Paired bootstrap differences:
Score-CAM minus Integrated Gradients: mean=0.0964, 95% CI=[0.0615, 0.1340]
Score-CAM minus Occlusion Sensitivity: mean=0.1671, 95% CI=[0.1252, 0.2103]
Integrated Gradients minus Occlusion Sensitivity: mean=0.0707, 95% CI=[0.0236, 0.1179]



### 3. Screening Reliability: Main Findings

Across 2,700 evaluations from 150 annotated chest X-ray images, 1,145 evaluations (42.41%) exhibited a localization alignment decrease of at least five percentage points.

Explanation instability demonstrated an overall ROC-AUC of 0.7911 (95% image-cluster bootstrap CI: 0.7671–0.8145) and average precision of 0.7231 (95% CI: 0.6705–0.7726).

At an explanation-similarity screening threshold of 0.80, 171 of the 1,145 substantial localization-loss evaluations (14.93%; 95% CI: 12.12–17.81%) were not flagged. Increasing the threshold to 0.90 reduced the missed-loss fraction to 3.41%, but increased the proportion of all evaluations flagged from 60.15% to 80.15%.

Screening performance differed across explanation methods:

- Integrated Gradients: ROC-AUC = 0.7850
- Occlusion Sensitivity: ROC-AUC = 0.7150
- Score-CAM: ROC-AUC = 0.8821

Paired image-cluster bootstrap comparisons showed positive ROC-AUC differences for Score-CAM relative to Integrated Gradients and Occlusion Sensitivity, and for Integrated Gradients relative to Occlusion Sensitivity. All three exploratory 95% confidence intervals excluded zero.

These findings indicate that explanation similarity provides useful information about localization deterioration, but screening performance depends on the explanation method and the selected similarity threshold.

A high similarity score does not guarantee preservation of localization alignment. Furthermore, discrimination of localization loss should not be interpreted as evidence of explanation faithfulness, clinical correctness, or diagnostic reliability.

The analysis is retrospective, uses repeated evaluations from 150 annotated positive images, and evaluates thresholds on the same dataset. Method comparisons are exploratory and have not been adjusted for multiple testing.


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score
import pandas as pd
import numpy as np

loss_thresholds = [0.03, 0.05, 0.10]
sensitivity_results = []

for loss_threshold in loss_thresholds:

    # Define substantial localization loss
    loss = df["alignment_change"] <= -loss_threshold

    # Instability score: higher means less stable
    instability = -df["explanation_similarity"]

    # Existing screening rule: flag similarity below 0.80
    flagged = df["explanation_similarity"] < 0.80

    # Screening outcomes
    tp = (flagged & loss).sum()
    fn = (~flagged & loss).sum()
    fp = (flagged & ~loss).sum()
    tn = (~flagged & ~loss).sum()

    sensitivity_results.append(
        {
            "loss_threshold": loss_threshold,
            "n_loss_cases": int(loss.sum()),
            "loss_prevalence_pct": 100 * loss.mean(),
            "roc_auc": roc_auc_score(loss.astype(int), instability),
            "average_precision": average_precision_score(loss.astype(int), instability),
            "missed_loss_cases_080": int(fn),
            "missed_loss_pct_080": 100 * fn / (tp + fn),
            "specificity_080": tn / (tn + fp),
        }
    )

loss_sensitivity = pd.DataFrame(sensitivity_results)

display(loss_sensitivity.round(4))

,loss_threshold,n_loss_cases,loss_prevalence_pct,roc_auc,average_precision,missed_loss_cases_080,missed_loss_pct_080,specificity_080
0,0.03,1355,50.1852,0.7811,0.7677,257,18.9668,0.6089
1,0.05,1145,42.4074,0.7911,0.7231,171,14.9345,0.5820
2,0.10,744,27.5556,0.8111,0.6129,60,8.0645,0.5194


In [ ]:
import pandas as pd
import numpy as np

# One box-area value per image
image_areas = df.groupby("sop_uid")["box_area_fraction"].agg(["min", "max"])

# Validate that box area is constant across evaluations
assert np.allclose(
    image_areas["min"], image_areas["max"], rtol=0, atol=1e-8
), "Box area varies across evaluations for an image."

# Divide unique images into three equally sized groups
image_areas["area_group"] = pd.qcut(
    image_areas["min"].rank(method="first"), q=3, labels=["Small", "Medium", "Large"]
)

# Map image-level groups back to evaluation rows
df["area_group"] = df["sop_uid"].map(image_areas["area_group"])

# Substantial localization loss at original 5pp threshold
loss = df["alignment_change"] <= -0.05

# Screening failures: substantial loss despite high similarity
missed = loss & (df["explanation_similarity"] >= 0.80)

area_results = []

for group_name, subset in df.groupby("area_group", observed=True):
    group_loss = subset["alignment_change"] <= -0.05

    group_missed = group_loss & (subset["explanation_similarity"] >= 0.80)

    area_results.append(
        {
            "area_group": str(group_name),
            "n_images": subset["sop_uid"].nunique(),
            "n_evaluations": len(subset),
            "median_box_area_fraction": subset["box_area_fraction"].median(),
            "loss_prevalence_pct": 100 * group_loss.mean(),
            "missed_loss_cases": int(group_missed.sum()),
            "missed_loss_pct": (
                100 * group_missed.sum() / group_loss.sum()
                if group_loss.sum()
                else np.nan
            ),
        }
    )

area_summary = pd.DataFrame(area_results)

display(area_summary.round(4))

,area_group,n_images,n_evaluations,median_box_area_fraction,loss_prevalence_pct,missed_loss_cases,missed_loss_pct
0,Small,50,900,0.0317,30.6667,26,9.4203
1,Medium,50,900,0.0803,44.5556,65,16.2095
2,Large,50,900,0.1997,52.0000,80,17.0940


In [ ]:
baseline_area_summary = (
    df.groupby(["method", "area_group"], observed=True)
    .agg(
        n_images=("sop_uid", "nunique"),
        median_box_area=("box_area_fraction", "median"),
        mean_original_alignment=("original_alignment", "mean"),
        median_original_alignment=("original_alignment", "median"),
        mean_alignment_change=("alignment_change", "mean"),
        median_alignment_change=("alignment_change", "median"),
    )
    .reset_index()
)

display(baseline_area_summary.round(4))

,method,area_group,n_images,median_box_area,mean_original_alignment,median_original_alignment,mean_alignment_change,median_alignment_change
0,Integrated Gradients,Small,50,0.0317,0.1569,0.1586,-0.0439,-0.0212
1,Integrated Gradients,Medium,50,0.0803,0.3027,0.2885,-0.0792,-0.0470
2,Integrated Gradients,Large,50,0.1997,0.5310,0.5414,-0.1155,-0.0681
3,Occlusion Sensitivity,Small,50,0.0317,0.1438,0.1418,-0.0380,-0.0182
4,Occlusion Sensitivity,Medium,50,0.0803,0.2897,0.2848,-0.0624,-0.0335
5,Occlusion Sensitivity,Large,50,0.1997,0.5473,0.5542,-0.0861,-0.0579
6,Score-CAM,Small,50,0.0317,0.1084,0.0850,-0.0227,-0.0087
7,Score-CAM,Medium,50,0.0803,0.2588,0.2505,-0.0660,-0.0226
8,Score-CAM,Large,50,0.1997,0.4907,0.4911,-0.1063,-0.0365


In [ ]:
print("Original alignment distribution:")

display(df["original_alignment"].describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.50]))

for threshold in [0.001, 0.01, 0.05]:
    count = (df["original_alignment"] < threshold).sum()
    print(f"Original alignment < {threshold}: " f"{count} evaluations")

Original alignment distribution:


,original_alignment
count,2700.000000
mean,0.314371
std,0.200039
min,0.009763
1%,0.014520
5%,0.037986
10%,0.064510
25%,0.153853
50%,0.282148
max,0.907960


Original alignment < 0.001: 0 evaluations
Original alignment < 0.01: 6 evaluations
Original alignment < 0.05: 192 evaluations


In [ ]:
# Relative localization loss sensitivity analysis

# Avoid unstable ratios when original alignment
# is extremely close to zero.
epsilon = 1e-6

valid = df["original_alignment"] > epsilon

relative_df = df.loc[valid].copy()

# Positive values indicate proportional alignment loss
relative_df["relative_alignment_loss"] = (
    -relative_df["alignment_change"] / relative_df["original_alignment"]
)

# Define substantial relative loss as at least 20%
relative_df["relative_loss_20pct"] = relative_df["relative_alignment_loss"] >= 0.20

relative_df["missed_relative_loss"] = relative_df["relative_loss_20pct"] & (
    relative_df["explanation_similarity"] >= 0.80
)

relative_area_summary = (
    relative_df.groupby("area_group", observed=True)
    .agg(
        n_evaluations=("sop_uid", "size"),
        median_relative_loss=("relative_alignment_loss", "median"),
        relative_loss_prevalence=("relative_loss_20pct", "mean"),
        n_relative_loss=("relative_loss_20pct", "sum"),
        missed_relative_loss=("missed_relative_loss", "sum"),
    )
    .reset_index()
)

relative_area_summary["missed_loss_pct"] = (
    100
    * relative_area_summary["missed_relative_loss"]
    / relative_area_summary["n_relative_loss"]
)

relative_area_summary["relative_loss_prevalence"] *= 100

print("Evaluations excluded due to near-zero baseline alignment:", (~valid).sum())

display(relative_area_summary.round(4))

Evaluations excluded due to near-zero baseline alignment: 0


,area_group,n_evaluations,median_relative_loss,relative_loss_prevalence,n_relative_loss,missed_relative_loss,missed_loss_pct
0,Small,900,0.1751,47.8889,431,73,16.9374
1,Medium,900,0.1474,44.8889,404,60,14.8515
2,Large,900,0.1062,39.3333,354,32,9.0395


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

relative_screening_results = []

for method, group in [("Overall", relative_df), *list(relative_df.groupby("method"))]:
    y_true = group["relative_loss_20pct"].astype(int)
    y_score = -group["explanation_similarity"]

    flagged = group["explanation_similarity"] < 0.80

    loss_cases = group["relative_loss_20pct"]
    missed = loss_cases & ~flagged

    relative_screening_results.append(
        {
            "method": method,
            "n_evaluations": len(group),
            "relative_loss_prevalence_pct": 100 * loss_cases.mean(),
            "roc_auc": roc_auc_score(y_true, y_score),
            "average_precision": average_precision_score(y_true, y_score),
            "missed_loss_cases_080": int(missed.sum()),
            "missed_loss_pct_080": (100 * missed.sum() / loss_cases.sum()),
        }
    )

relative_screening_summary = pd.DataFrame(relative_screening_results)

display(relative_screening_summary.round(4))

,method,n_evaluations,relative_loss_prevalence_pct,roc_auc,average_precision,missed_loss_cases_080,missed_loss_pct_080
0,Overall,2700,44.0370,0.8251,0.7847,165,13.8772
1,Integrated Gradients,900,45.6667,0.8391,0.8065,12,2.9197
2,Occlusion Sensitivity,900,43.6667,0.7759,0.6565,77,19.5929
3,Score-CAM,900,42.7778,0.9041,0.8816,76,19.7403


In [ ]:
# Sensitivity to small original-alignment denominators

from sklearn.metrics import roc_auc_score, average_precision_score

baseline_checks = []

for minimum_alignment in [0, 0.01, 0.05]:
    subset = relative_df[relative_df["original_alignment"] > minimum_alignment].copy()

    y_true = subset["relative_loss_20pct"].astype(int)
    y_score = -subset["explanation_similarity"]

    missed = subset["relative_loss_20pct"] & (subset["explanation_similarity"] >= 0.80)

    baseline_checks.append(
        {
            "minimum_alignment": minimum_alignment,
            "n_evaluations": len(subset),
            "relative_loss_prevalence_pct": (100 * y_true.mean()),
            "roc_auc": roc_auc_score(y_true, y_score),
            "average_precision": average_precision_score(y_true, y_score),
            "missed_loss_pct_080": (100 * missed.sum() / y_true.sum()),
        }
    )

baseline_sensitivity = pd.DataFrame(baseline_checks)

display(baseline_sensitivity.round(4))

,minimum_alignment,n_evaluations,relative_loss_prevalence_pct,roc_auc,average_precision,missed_loss_pct_080
0,0.00,2700,44.0370,0.8251,0.7847,13.8772
1,0.01,2694,44.1351,0.8259,0.7860,13.8772
2,0.05,2508,44.6571,0.8463,0.8219,12.6786


In [ ]:
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n_bootstrap = 2000

# Use the complete dataset; all original alignments
# were confirmed to be above 1e-6.
bootstrap_df = df.copy()

bootstrap_df["absolute_loss"] = bootstrap_df["alignment_change"] <= -0.05

bootstrap_df["relative_loss"] = (
    -bootstrap_df["alignment_change"] / bootstrap_df["original_alignment"] >= 0.20
)

bootstrap_df["instability_score"] = -bootstrap_df["explanation_similarity"]

image_groups = {uid: group for uid, group in bootstrap_df.groupby("sop_uid")}

image_ids = np.array(list(image_groups.keys()))

bootstrap_metrics = []

for _ in range(n_bootstrap):

    sampled_ids = rng.choice(image_ids, size=len(image_ids), replace=True)

    sample = pd.concat([image_groups[uid] for uid in sampled_ids], ignore_index=True)

    y_absolute = sample["absolute_loss"].astype(int)
    y_relative = sample["relative_loss"].astype(int)
    scores = sample["instability_score"]

    if y_absolute.nunique() < 2 or y_relative.nunique() < 2:
        continue

    auc_absolute = roc_auc_score(y_absolute, scores)
    auc_relative = roc_auc_score(y_relative, scores)

    bootstrap_metrics.append(
        {
            "absolute_auc": auc_absolute,
            "relative_auc": auc_relative,
            "auc_difference": auc_relative - auc_absolute,
        }
    )

paired_bootstrap = pd.DataFrame(bootstrap_metrics)

summary = pd.DataFrame(
    {
        "Metric": [
            "Absolute-loss ROC-AUC",
            "Relative-loss ROC-AUC",
            "Relative minus absolute ROC-AUC",
        ],
        "Bootstrap mean": [
            paired_bootstrap["absolute_auc"].mean(),
            paired_bootstrap["relative_auc"].mean(),
            paired_bootstrap["auc_difference"].mean(),
        ],
        "CI lower": [
            paired_bootstrap["absolute_auc"].quantile(0.025),
            paired_bootstrap["relative_auc"].quantile(0.025),
            paired_bootstrap["auc_difference"].quantile(0.025),
        ],
        "CI upper": [
            paired_bootstrap["absolute_auc"].quantile(0.975),
            paired_bootstrap["relative_auc"].quantile(0.975),
            paired_bootstrap["auc_difference"].quantile(0.975),
        ],
    }
)

print("Valid bootstrap samples:", len(paired_bootstrap))
display(summary.round(4))

Valid bootstrap samples: 2000


,Metric,Bootstrap mean,CI lower,CI upper
0,Absolute-loss ROC-AUC,0.7914,0.7671,0.8145
1,Relative-loss ROC-AUC,0.8255,0.8018,0.8484
2,Relative minus absolute ROC-AUC,0.0341,0.0182,0.0521



## 4. Localization Sensitivity: Main Findings

### 4.1 Absolute Localization-Loss Thresholds

The stability-based screening analysis was repeated using absolute localization alignment decreases of at least 3, 5, and 10 percentage points.

ROC-AUC ranged from 0.7811 to 0.8111 across these definitions. At the explanation-similarity screening threshold of 0.80, the percentage of substantial localization-loss evaluations missed by screening ranged from 18.97% for the 3-percentage-point definition to 8.06% for the 10-percentage-point definition.

Thus, the screening limitation was not exclusive to the original 5-percentage-point threshold, although its magnitude depended on the selected outcome definition.

### 4.2 Annotated Opacity-Region Size

The 150 annotated images were divided into three equal-sized groups according to the fraction of image area covered by the union of opacity bounding boxes.

Under the original absolute-loss definition, substantial localization-loss prevalence increased from 30.67% in the small-region group to 52.00% in the large-region group.

However, original localization alignment also increased substantially with annotated-region size across all three explanation methods. This suggests that baseline alignment and the scale of absolute changes may contribute to the observed area-dependent pattern.

### 4.3 Relative Localization Deterioration

A complementary analysis defined relative localization loss as the decrease in alignment divided by the original alignment. A relative decrease of at least 20% was considered substantial for this exploratory analysis.

No evaluations were excluded due to near-zero original alignment.

Under this definition, substantial-loss prevalence decreased from 47.89% for small annotated regions to 39.33% for large regions, reversing the direction observed with the absolute-loss definition.

The percentage of substantial losses missed by the similarity-based screening rule also varied across region sizes, with the highest missed-loss percentage occurring in the small-region group under the relative definition.

This reversal demonstrates that conclusions about localization deterioration are sensitive to the combination of metric definition and threshold.


The reversal should not be interpreted as evidence that smaller annotated regions are inherently more vulnerable to corruption. Absolute and relative loss capture different aspects of deterioration, and the selected thresholds are not directly equivalent. The observed differences may also reflect variation in original localization alignment.


### 4.4 Screening Performance Under Relative Loss

Using the relative-loss definition, explanation instability achieved an overall ROC-AUC of 0.8251 and average precision of 0.7847.

Method-specific ROC-AUC values were:

- Integrated Gradients: 0.8391
- Occlusion Sensitivity: 0.7759
- Score-CAM: 0.9041

At the original similarity screening threshold of 0.80, 165 evaluations with substantial relative localization loss were not flagged, corresponding to 13.88% of relative-loss cases.

These findings indicate that explanation similarity retains discriminatory information under the alternative outcome definition. However, high similarity still does not guarantee localization preservation.

### 4.5 Interpretation and Limitations

Absolute and relative localization-loss measures capture different aspects of deterioration. Neither should be interpreted as a clinically validated measure of explanation quality.

The absolute and relative thresholds were selected for exploratory analysis and do not represent equivalent levels of deterioration.

The observed associations with opacity-region size may reflect differences in baseline alignment, image characteristics, or other unmeasured factors. They should not be interpreted causally.

The analysis is restricted to annotated positive chest X-ray images and repeated evaluations of three explanation methods. It does not establish explanation faithfulness, clinical utility, or generalizability to other datasets and models.


### 4.6 Sensitivity to Low Original Localization Alignment

Because relative localization loss divides the alignment decrease by original alignment, evaluations with small baseline alignment may produce large proportional changes.

To investigate this concern, the relative-loss screening analysis was repeated after excluding evaluations with original alignment at or below 0.01 and 0.05.

The full dataset contained 2,700 evaluations, with a screening ROC-AUC of 0.8251 and a missed-loss rate of 13.88% at the similarity threshold of 0.80.

After excluding evaluations with original alignment at or below 0.01, 2,694 evaluations remained. ROC-AUC was 0.8259, and the missed-loss rate remained 13.88%.

After excluding evaluations with original alignment at or below 0.05, 2,508 evaluations remained. ROC-AUC increased to 0.8463, while the missed-loss rate decreased to 12.68%.

These findings indicate that the overall screening results are not driven exclusively by evaluations with low original localization alignment. However, screening performance changes somewhat when low-baseline evaluations are excluded.

The exclusion thresholds are exploratory, and the resulting subsets represent different evaluation populations. Therefore, differences in screening performance should not be interpreted as improvements in the underlying explanation methods.




## 5. Overall Conclusions

This notebook investigated two questions arising from the stability–localization disagreement audit:

1. Can explanation similarity serve as a retrospective screening signal for substantial localization deterioration?
2. How sensitive are screening conclusions to the definition of localization loss and the size of annotated opacity regions?

### 5.1 Stability as a Screening Signal


The reported ROC-AUC values characterize discrimination across repeated image–method–corruption evaluations, rather than performance on independent patients. Image-cluster bootstrap resampling was used to estimate uncertainty while preserving evaluations associated with the same image.


Across 2,700 evaluations from 150 annotated positive chest X-ray images, explanation instability showed moderate-to-strong discrimination for substantial absolute localization loss, with an overall ROC-AUC of 0.7911 (95% image-cluster bootstrap CI: 0.7671–0.8145).

However, screening was imperfect. At a similarity threshold of 0.80, 171 of 1,145 evaluations with an absolute alignment decrease of at least 5 percentage points were not flagged (14.93%).

Screening behavior also differed across explanation methods. Integrated Gradients, Occlusion Sensitivity, and Score-CAM exhibited different ROC-AUC values and missed-loss rates under the same screening rule.

These findings support treating explanation similarity as a potentially informative retrospective screening signal rather than a guarantee of localization preservation.

### 5.2 Sensitivity to Localization-Loss Definitions

The analysis considered several absolute alignment-loss thresholds and an alternative relative-loss definition.

Under the absolute-loss definition, evaluations involving larger annotated opacity regions showed higher substantial-loss prevalence. Under the relative-loss definition, the pattern reversed, with smaller annotated regions showing higher substantial-loss prevalence.

This indicates that conclusions about localization deterioration depend partly on the chosen metric and threshold.

For relative localization loss of at least 20%, explanation instability achieved an overall ROC-AUC of 0.8251 (95% image-cluster bootstrap CI: 0.8018–0.8484).

The observed ROC-AUC difference between relative and absolute loss was 0.0340. A paired image-cluster bootstrap estimated a mean difference of 0.0341 (95% CI: 0.0182–0.0521).

Although the relative-loss outcome produced higher observed discrimination, the two definitions identify different deterioration events. The difference should not be interpreted as evidence that one definition is inherently more valid.

At the similarity threshold of 0.80, 165 evaluations with substantial relative localization loss were not flagged (13.88%).

### 5.3 Implications for Explanation Reliability

The results reinforce the central finding of this project: high similarity between explanations does not necessarily imply preservation of localization within annotated opacity regions.

The observed disagreement is not limited to a single absolute-loss threshold. It remains relevant when deterioration is measured proportionally.

Furthermore, the apparent relationship between localization deterioration and annotated-region size changes across outcome definitions.

Together, these findings demonstrate why explanation reliability should not be summarized using explanation similarity alone.

### 5.4 Limitations

Several limitations constrain interpretation:

- The analysis used one trained ResNet-50 model and one chest X-ray dataset.
- Localization was measured using attribution mass within radiologist-annotated opacity boxes, which is an imperfect proxy for explanation quality.
- The analysis included repeated evaluations of the same images; image-cluster bootstrapping was used to account for this dependence in uncertainty estimates.
- The similarity and localization-loss thresholds were exploratory rather than clinically validated.
- The absolute and relative deterioration definitions are not equivalent.
- Method-specific differences may reflect properties of the attribution methods, the fixed explanation targets, or the evaluation design.
- Screening performance was assessed retrospectively on the same dataset used to explore the screening thresholds. It should not be interpreted as prospectively validated performance.
- Neither high localization alignment nor explanation stability establishes faithfulness to the model's decision-making process or clinical usefulness.

### 5.5 Final Takeaway

Explanation similarity contains useful information about localization deterioration, but it is not a sufficient reliability criterion.

Both absolute and relative localization-loss analyses reveal cases where explanations remain comparatively stable while attribution alignment with annotated opacity regions deteriorates.

The magnitude and distribution of these failures depend on the explanation method, the screening threshold, and the localization outcome definition.

A more comprehensive evaluation of explanation reliability therefore requires examining both explanation stability and task-relevant localization behavior, while acknowledging the limitations of each metric.
